## Imports

In [36]:
import os
import torch
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

hf_token = os.getenv("HF_TOKEN")

print("Imports complete and HF token loaded.")


Imports complete and HF token loaded.


## File Type Check

In [37]:
def get_document_loader(file_path):
    """
    Dynamically selects the correct LangChain document loader based on file extension.
    """
    # Get the file extension
    _, extension = os.path.splitext(file_path)
    extension = extension.lower()

    if extension == ".pdf":
        print("Loading PDF file...")
        return PyPDFLoader(file_path)
    elif extension == ".txt":
        print("Loading text file...")
        return TextLoader(file_path)
    elif extension == ".docx":
        print("Loading DOCX file...")
        return Docx2txtLoader(file_path)
    else:
        raise ValueError(f"Unsupported file type: {extension}")


## Data Ingestion & Chunking

In [38]:
# In LangChain, document is load and then chunked using Text Splitter.
file_path = "sample_doc.txt"

# Load the document using a document loader
loader = get_document_loader(file_path)
documents = loader.load()

# Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(documents)

print(f"Successfully loaded and split content from {file_path}.")
print(f"Number of document chunks: {len(docs)}")


Loading text file...
Successfully loaded and split content from sample_doc.txt.
Number of document chunks: 2


## Embeddings

In [39]:
model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device": "cpu"} # Use "cuda" if you have a GPU
encode_kwargs = {"normalize_embeddings": True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("BGE embedding model configured successfully.")

BGE embedding model configured successfully.


## Vector Store

In [40]:
vectorstore = FAISS.from_documents(docs, embeddings)

print("Vector store has been built successfully.")

Vector store has been built successfully.


## Llama 3.1 8B Instruct

In [41]:
llm = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Meta-Llama-3.1-8B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 512,
        "temperature": 0.1,
        "repetition_penalty": 1.15 # Prevent repeated phrases
    },
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    },
)

print("Llama 3.1 8B Instruct model configured successfully.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama 3.1 8B Instruct model configured successfully.


# Query

In [42]:
template = """
You are a helpful assistant. Use the following context to answer the question at the end.
Answer in a clear and conversational manner. Do not just copy the context.
Synthesize the information to respond directly to the question.

Context:
{context}

Question: {input}

Answer:
"""
prompt = PromptTemplate.from_template(template)

retriever = vectorstore.as_retriever()

rag_chain = create_retrieval_chain(
    retriever,
    create_stuff_documents_chain(llm, prompt)
)

print("Modern RAG chain created successfully.")

Modern RAG chain created successfully.


## Example Usage

In [ ]:
query_text = "How do I reset my password?" 
# Retrieve relevant documents from the vector store
similar_docs = vectorstore.similarity_search(query_text)

# Run the chain with the retrieved documents and the query
response = rag_chain.invoke({"input": query_text})

print(response["answer"])


You are a helpful assistant. Use the following context to answer the question at the end.
Answer in a clear and conversational manner. Do not just copy the context.
Synthesize the information to respond directly to the question.

Context:
## Password Reset Functionality
To implement password reset in your application:
1. Create a password reset endpoint: POST /api/auth/reset-password
2. The system will send a reset token to the user's email
3. Users can then use the token to set a new password via PUT /api/auth/update-password
4. Tokens expire after 24 hours for security reasons

## Troubleshooting Common Issues
- API calls failing with 401 errors: Check your API key configuration
- Database connection timeouts: Increase the timeout settings in database.js
- SSL certificate errors in production: Ensure proper certificate installation

# TechFramework API Documentation

## Overview
TechFramework is a modern web development framework designed for building scalable applications. 
It supp

: 